# ✦ InkCluster — Transformer Ink Recognizer
### Training on MathWriting Dataset (Google, 2024)

**Pipeline:**
```
InkML strokes → Delta features → StrokeEncoder → CrossStrokeTransformer → CTC → LaTeX string
```

**This notebook:**
1. Downloads MathWriting dataset (InkML format, real stroke x/y/t data)
2. Parses InkML → stroke sequences
3. Converts to delta features [Δx, Δy, sinθ, cosθ, pen_lift]
4. Trains Transformer + CTC model
5. Evaluates with CER (Character Error Rate)
6. Exports to ONNX for browser deployment

> **Runtime:** GPU (T4 recommended). Runtime → Change runtime type → T4 GPU

## 0. Install & Imports

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q editdistance tqdm

In [ ]:
import os, math, tarfile, urllib.request, glob, random
import xml.etree.ElementTree as ET
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import editdistance
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Download MathWriting Dataset

In [ ]:
# ── Choose which split to download ──
# Start with EXCERPT (1.5 MB) to test pipeline quickly
# Switch to FULL (2.9 GB) for real training

USE_EXCERPT = True   # ← set False for full dataset

if USE_EXCERPT:
    URL  = 'https://storage.googleapis.com/mathwriting_data/mathwriting-2024-excerpt.tgz'
    FILE = 'mathwriting-excerpt.tgz'
else:
    URL  = 'https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz'
    FILE = 'mathwriting-full.tgz'

DATA_DIR = './mathwriting'

if not os.path.exists(DATA_DIR):
    print(f'Downloading {FILE} ...')
    urllib.request.urlretrieve(URL, FILE)
    print('Extracting ...')
    with tarfile.open(FILE, 'r:gz') as tar:
        tar.extractall('.')
    print('Done!')
else:
    print('Dataset already downloaded.')

# Find all inkml files
all_inkml = glob.glob(os.path.join(DATA_DIR, '**', '*.inkml'), recursive=True)
print(f'Found {len(all_inkml)} InkML files')

## 2. Parse InkML → Strokes

In [ ]:
# ── InkML format (MathWriting) ──
# Each file contains:
#   <ink>
#     <annotation type='truth'>\frac{a}{b}</annotation>  ← ground truth LaTeX
#     <trace>x1 y1,x2 y2,...</trace>   ← one stroke (x y pairs)
#     <trace>...</trace>
#   </ink>

NS = {'ink': 'http://www.w3.org/2003/InkML'}

def parse_inkml(filepath):
    """
    Parse one InkML file.
    Returns:
        strokes : list of np.arrays, each shape [N_points, 2] (x, y)
        label   : ground truth string (LaTeX for MathWriting)
    """
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()

        # Get ground truth label
        label = ''
        for ann in root.findall('.//ink:annotation', NS):
            if ann.get('type') == 'truth':
                label = ann.text or ''
                break
        # Fallback: try without namespace
        if not label:
            for ann in root.findall('.//annotation'):
                if ann.get('type') == 'truth':
                    label = ann.text or ''
                    break

        # Parse strokes
        strokes = []
        traces = root.findall('.//ink:trace', NS) or root.findall('.//trace')

        for trace in traces:
            text = trace.text
            if not text: continue
            points = []
            for pt in text.strip().split(','):
                coords = pt.strip().split()
                if len(coords) >= 2:
                    try:
                        x, y = float(coords[0]), float(coords[1])
                        points.append([x, y])
                    except ValueError:
                        continue
            if len(points) >= 2:
                strokes.append(np.array(points, dtype=np.float32))

        return strokes, label

    except Exception as e:
        return [], ''


# Test on first file
sample_file = all_inkml[0]
strokes, label = parse_inkml(sample_file)
print(f'Sample file: {os.path.basename(sample_file)}')
print(f'Label: "{label}"')
print(f'Strokes: {len(strokes)}')
print(f'First stroke shape: {strokes[0].shape}')
print(f'First stroke (first 3 pts): {strokes[0][:3]}')

## 3. Feature Extraction — Delta Encoding

In [ ]:
# ── Delta Feature Encoding ──
# Why delta (relative) instead of absolute coordinates:
#   - Translation invariant: "Hello" at (0,0) and (500,500) → same features
#   - Scale invariant after normalisation: large/small writing → same features
#   - Standard in all ink recognition research (SketchRNN, Google Ink)

def normalise_strokes(strokes):
    """
    Normalise all strokes so the bounding box height = 1.0
    and center = (0, 0). This makes the model scale-invariant.
    """
    all_pts = np.concatenate(strokes, axis=0)  # [total_pts, 2]
    x_min, y_min = all_pts.min(axis=0)
    x_max, y_max = all_pts.max(axis=0)

    height = max(y_max - y_min, 1e-6)  # avoid div by zero
    cx = (x_min + x_max) / 2
    cy = (y_min + y_max) / 2

    normed = []
    for s in strokes:
        s_norm = s.copy()
        s_norm[:, 0] = (s_norm[:, 0] - cx) / height
        s_norm[:, 1] = (s_norm[:, 1] - cy) / height
        normed.append(s_norm)
    return normed


def strokes_to_delta_features(strokes, max_points_per_stroke=64):
    """
    Convert strokes to delta feature matrix.

    Each point → 5 features:
        [Δx, Δy, sin(θ), cos(θ), pen_lift]

    where:
        Δx, Δy   = displacement from previous point
        θ        = direction angle
        pen_lift = 1.0 at last point of each stroke (pen leaves paper)

    Returns:
        features : np.array [N_strokes, max_points_per_stroke, 5]
        mask     : np.array [N_strokes, max_points_per_stroke]  (1=valid, 0=padding)
    """
    if not strokes:
        return None, None

    strokes = normalise_strokes(strokes)
    n_strokes = len(strokes)
    features = np.zeros((n_strokes, max_points_per_stroke, 5), dtype=np.float32)
    mask     = np.zeros((n_strokes, max_points_per_stroke),    dtype=np.float32)

    for si, stroke in enumerate(strokes):
        pts = stroke[:max_points_per_stroke]  # truncate if too long
        n   = len(pts)

        for pi in range(n):
            if pi == 0:
                dx, dy = 0.0, 0.0
            else:
                dx = pts[pi, 0] - pts[pi-1, 0]
                dy = pts[pi, 1] - pts[pi-1, 1]

            angle = math.atan2(dy, dx)
            pen_lift = 1.0 if pi == n - 1 else 0.0

            features[si, pi] = [dx, dy, math.sin(angle), math.cos(angle), pen_lift]
            mask[si, pi] = 1.0

    return features, mask


# Test
feats, mask = strokes_to_delta_features(strokes)
print(f'Feature shape: {feats.shape}  → [n_strokes, max_pts, 5_features]')
print(f'Mask shape:    {mask.shape}')
print(f'Active points per stroke: {mask.sum(axis=1).astype(int)}')
print(f'Sample features (stroke 0, pts 0-2):')
print(f'  [Δx,    Δy,    sinθ,   cosθ,  pen]')
print(feats[0, :3].round(4))

## 4. Vocabulary & Label Encoding

In [ ]:
# ── Vocabulary ──
# MathWriting labels are LaTeX strings like \frac{a}{b}
# We tokenise at CHARACTER level — each char is one token
# This is simpler than subword tokenisation and works well for CTC

# Scan all files to build vocabulary
print('Building vocabulary from all labels...')

all_labels = []
valid_files = []

for fp in tqdm(all_inkml):
    strokes_i, label_i = parse_inkml(fp)
    if len(strokes_i) >= 1 and len(label_i) >= 1:
        all_labels.append(label_i)
        valid_files.append(fp)

print(f'Valid samples: {len(valid_files)} / {len(all_inkml)}')

# Build character vocabulary
all_chars = sorted(set(''.join(all_labels)))
VOCAB = ['<PAD>'] + all_chars + ['<BLANK>']  # BLANK is for CTC
BLANK_IDX = len(VOCAB) - 1
PAD_IDX   = 0

char2idx = {c: i for i, c in enumerate(VOCAB)}
idx2char = {i: c for i, c in enumerate(VOCAB)}

print(f'Vocabulary size: {len(VOCAB)} chars')
print(f'BLANK index: {BLANK_IDX}')
print(f'Sample chars: {all_chars[:30]}')


def encode_label(text):
    """Convert string to list of token indices."""
    return [char2idx[c] for c in text if c in char2idx]

def decode_label(indices):
    """Convert token indices back to string, skipping PAD and BLANK."""
    return ''.join(idx2char[i] for i in indices
                   if i not in (PAD_IDX, BLANK_IDX) and i in idx2char)

# Test
sample_label = all_labels[0]
encoded = encode_label(sample_label)
decoded = decode_label(encoded)
print(f'\nSample label:  "{sample_label}"')
print(f'Encoded:       {encoded}')
print(f'Decoded back:  "{decoded}"')
print(f'Match: {sample_label == decoded}')

## 5. Dataset & DataLoader

In [ ]:
MAX_STROKES = 32   # max strokes per sample (truncate longer)
MAX_PTS     = 64   # max points per stroke
MAX_LABEL   = 128  # max label length


class InkDataset(Dataset):
    def __init__(self, file_list):
        self.files = file_list
        self.cache = {}  # cache parsed samples

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        if idx in self.cache:
            return self.cache[idx]

        fp = self.files[idx]
        strokes, label = parse_inkml(fp)

        # Truncate strokes
        strokes = strokes[:MAX_STROKES]

        # Get features
        feats, stroke_mask = strokes_to_delta_features(strokes, MAX_PTS)

        # Encode label
        label_enc = encode_label(label)[:MAX_LABEL]

        sample = {
            'features':     torch.FloatTensor(feats),          # [S, P, 5]
            'stroke_mask':  torch.FloatTensor(stroke_mask),    # [S, P]
            'label':        torch.LongTensor(label_enc),       # [L]
            'n_strokes':    len(strokes),
            'label_len':    len(label_enc),
            'label_str':    label,
        }
        self.cache[idx] = sample
        return sample


def collate_fn(batch):
    """Pad batch to same size for DataLoader."""
    max_s = max(b['n_strokes'] for b in batch)
    max_l = max(b['label_len'] for b in batch)

    features    = torch.zeros(len(batch), max_s, MAX_PTS, 5)
    stroke_mask = torch.zeros(len(batch), max_s, MAX_PTS)
    labels      = torch.zeros(len(batch), max_l, dtype=torch.long)
    n_strokes   = torch.zeros(len(batch), dtype=torch.long)
    label_lens  = torch.zeros(len(batch), dtype=torch.long)

    for i, b in enumerate(batch):
        s = b['n_strokes']
        l = b['label_len']
        features[i, :s]    = b['features']
        stroke_mask[i, :s] = b['stroke_mask']
        labels[i, :l]      = b['label']
        n_strokes[i]       = s
        label_lens[i]      = l

    return {
        'features':    features,      # [B, S, P, 5]
        'stroke_mask': stroke_mask,   # [B, S, P]
        'labels':      labels,        # [B, L]
        'n_strokes':   n_strokes,     # [B]
        'label_lens':  label_lens,    # [B]
        'label_strs':  [b['label_str'] for b in batch],
    }


# ── Split train/val ──
random.shuffle(valid_files)
split = int(0.9 * len(valid_files))
train_files = valid_files[:split]
val_files   = valid_files[split:]

train_ds = InkDataset(train_files)
val_ds   = InkDataset(val_files)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples  |  Val: {len(val_ds)} samples')
print(f'Train batches: {len(train_loader)}')

# Verify a batch
batch = next(iter(train_loader))
print(f'\nSample batch:')
print(f'  features shape:  {batch["features"].shape}')  # [B, S, P, 5]
print(f'  labels shape:    {batch["labels"].shape}')     # [B, L]
print(f'  first label:     "{batch["label_strs"][0]}"')

## 6. Model — Transformer + CTC

In [ ]:
def make_pos_enc(max_len, d_model, device):
    """Sinusoidal positional encoding."""
    pe  = torch.zeros(max_len, d_model, device=device)
    pos = torch.arange(max_len, device=device).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2, device=device).float()
                    * -(math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe.unsqueeze(0)  # [1, max_len, d_model]


class InkTransformer(nn.Module):
    """
    Two-stage Transformer for online handwriting recognition.

    Stage 1 — StrokeEncoder:
        Processes each stroke's point sequence independently.
        Shared weights across all strokes in the input.
        Output: one embedding vector per stroke.

    Stage 2 — WordTransformer:
        Looks at all stroke embeddings together.
        Models relationships between strokes (word-level context).
        Output: contextualised stroke embeddings.

    Stage 3 — CTC Head:
        Linear projection to vocabulary size.
        CTC loss during training, beam search at inference.
    """

    def __init__(
        self,
        vocab_size,
        d_model      = 128,
        n_stroke_heads = 4,
        n_word_heads   = 8,
        n_stroke_layers= 2,
        n_word_layers  = 4,
        dropout        = 0.1,
        max_strokes    = 64,
        max_pts        = 64,
    ):
        super().__init__()
        self.d_model     = d_model
        self.max_strokes = max_strokes
        self.max_pts     = max_pts

        # ── Stage 1: per-stroke encoder ──
        self.stroke_proj = nn.Linear(5, d_model)
        stroke_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_stroke_heads,
            dim_feedforward=d_model*2, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.stroke_enc = nn.TransformerEncoder(stroke_layer, n_stroke_layers)

        # ── Stage 2: cross-stroke transformer ──
        word_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_word_heads,
            dim_feedforward=d_model*4, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.word_enc = nn.TransformerEncoder(word_layer, n_word_layers)

        # ── Stage 3: CTC head ──
        self.norm    = nn.LayerNorm(d_model)
        self.ctc_head = nn.Linear(d_model, vocab_size)
        self.dropout  = nn.Dropout(dropout)

    def encode_strokes(self, x, stroke_mask):
        """
        x:           [B, S, P, 5]   — batch, strokes, points, features
        stroke_mask: [B, S, P]      — 1=valid point, 0=padding
        returns:     [B, S, d_model] — one embedding per stroke
        """
        B, S, P, F = x.shape

        # Process all strokes in parallel: reshape to [B*S, P, F]
        x    = x.view(B*S, P, F)
        mask = stroke_mask.view(B*S, P)  # [B*S, P]

        # Project input features to d_model
        x = self.stroke_proj(x)   # [B*S, P, d_model]

        # Add positional encoding
        x = x + make_pos_enc(P, self.d_model, x.device)

        # Create attention mask (True = IGNORE this position)
        key_mask = (mask == 0)    # [B*S, P]

        # Transformer over points within each stroke
        x = self.stroke_enc(x, src_key_padding_mask=key_mask)  # [B*S, P, d]

        # Mean-pool over valid points only
        mask_exp = mask.unsqueeze(-1)         # [B*S, P, 1]
        x = (x * mask_exp).sum(dim=1)         # [B*S, d]
        x = x / mask_exp.sum(dim=1).clamp(min=1)  # normalise

        return x.view(B, S, self.d_model)     # [B, S, d_model]

    def forward(self, features, stroke_mask, n_strokes):
        """
        features:   [B, S, P, 5]
        stroke_mask:[B, S, P]
        n_strokes:  [B] — actual number of strokes per sample
        returns:    [B, S, vocab_size] log-probs (for CTC)
        """
        B, S = features.shape[:2]

        # Stage 1
        stroke_emb = self.encode_strokes(features, stroke_mask)  # [B, S, d]
        stroke_emb = self.dropout(stroke_emb)

        # Add stroke-level positional encoding
        stroke_emb = stroke_emb + make_pos_enc(S, self.d_model, stroke_emb.device)

        # Stroke-level padding mask for cross-stroke attention
        stroke_pad_mask = torch.zeros(B, S, dtype=torch.bool, device=features.device)
        for i, ns in enumerate(n_strokes):
            stroke_pad_mask[i, ns:] = True  # mask out padded strokes

        # Stage 2
        word_emb = self.word_enc(
            stroke_emb,
            src_key_padding_mask=stroke_pad_mask
        )  # [B, S, d]

        # Stage 3
        out = self.norm(word_emb)
        out = self.ctc_head(out)             # [B, S, vocab]
        return torch.log_softmax(out, dim=-1)


# ── Instantiate model ──
model = InkTransformer(
    vocab_size     = len(VOCAB),
    d_model        = 128,
    n_stroke_heads = 4,
    n_word_heads   = 8,
    n_stroke_layers= 2,
    n_word_layers  = 4,
    dropout        = 0.1,
    max_strokes    = MAX_STROKES,
    max_pts        = MAX_PTS,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}  (~{total_params/1e6:.1f}M)')

# Quick forward pass test
with torch.no_grad():
    test_feats = batch['features'].to(device)
    test_mask  = batch['stroke_mask'].to(device)
    test_ns    = batch['n_strokes'].to(device)
    test_out   = model(test_feats, test_mask, test_ns)
    print(f'Forward pass output shape: {test_out.shape}')  # [B, S, vocab]
    print(f'Output is log-probs (should be ≤ 0): max={test_out.max():.3f}')

## 7. CTC Decoder (Greedy + Beam Search)

In [ ]:
def ctc_greedy_decode(log_probs, blank_idx):
    """
    Greedy CTC decoding.
    log_probs: [T, vocab_size] — log probabilities over time
    Returns: decoded string
    """
    # Take argmax at each time step
    indices = log_probs.argmax(dim=-1).tolist()  # [T]

    # CTC collapse: remove consecutive duplicates, then remove blanks
    collapsed = []
    prev = None
    for idx in indices:
        if idx != prev:
            collapsed.append(idx)
            prev = idx

    # Remove blank tokens
    tokens = [i for i in collapsed if i != blank_idx and i != PAD_IDX]
    return decode_label(tokens)


def decode_batch(log_probs, n_strokes):
    """
    Decode a whole batch.
    log_probs: [B, S, vocab]
    n_strokes: [B]
    Returns: list of decoded strings
    """
    results = []
    for i in range(log_probs.shape[0]):
        ns = n_strokes[i].item()
        lp = log_probs[i, :ns]   # [valid_strokes, vocab]
        results.append(ctc_greedy_decode(lp, BLANK_IDX))
    return results


def compute_cer(predictions, targets):
    """
    Character Error Rate = edit_distance / len(target)
    Lower is better. 0.0 = perfect, 1.0 = completely wrong.
    """
    total_dist = 0
    total_len  = 0
    for pred, tgt in zip(predictions, targets):
        total_dist += editdistance.eval(pred, tgt)
        total_len  += max(len(tgt), 1)
    return total_dist / total_len


print('Decoder functions ready.')

## 8. Training Loop

In [ ]:
# ── Hyperparameters ──
EPOCHS    = 50      # increase to 100 for full dataset
LR        = 3e-4
CLIP_GRAD = 1.0
SAVE_PATH = 'ink_model_best.pt'

ctc_loss  = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True, reduction='mean')
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_cer = float('inf')
history = {'train_loss': [], 'val_loss': [], 'val_cer': []}


def train_epoch(loader):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc='Train', leave=False):
        features   = batch['features'].to(device)
        stroke_mask= batch['stroke_mask'].to(device)
        labels     = batch['labels'].to(device)
        n_strokes  = batch['n_strokes'].to(device)
        label_lens = batch['label_lens'].to(device)

        # Forward
        log_probs = model(features, stroke_mask, n_strokes)  # [B, S, V]

        # CTC expects [T, B, V]
        log_probs_ctc = log_probs.permute(1, 0, 2)  # [S, B, V]

        # Input lengths = number of valid strokes per sample
        input_lengths = n_strokes

        loss = ctc_loss(log_probs_ctc, labels, input_lengths, label_lens)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(loader):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_tgts   = []

    for batch in tqdm(loader, desc='Val  ', leave=False):
        features   = batch['features'].to(device)
        stroke_mask= batch['stroke_mask'].to(device)
        labels     = batch['labels'].to(device)
        n_strokes  = batch['n_strokes'].to(device)
        label_lens = batch['label_lens'].to(device)

        log_probs = model(features, stroke_mask, n_strokes)
        log_probs_ctc = log_probs.permute(1, 0, 2)

        loss = ctc_loss(log_probs_ctc, labels, n_strokes, label_lens)
        total_loss += loss.item()

        # Decode predictions
        preds = decode_batch(log_probs.cpu(), n_strokes.cpu())
        all_preds.extend(preds)
        all_tgts.extend(batch['label_strs'])

    cer = compute_cer(all_preds, all_tgts)
    return total_loss / len(loader), cer, all_preds[:5], all_tgts[:5]


print('Starting training...')
print(f'Epochs: {EPOCHS}  |  LR: {LR}  |  Device: {device}')
print('─' * 60)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(train_loader)
    val_loss, val_cer, sample_preds, sample_tgts = val_epoch(val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cer'].append(val_cer)

    # Save best model
    if val_cer < best_val_cer:
        best_val_cer = val_cer
        torch.save({
            'epoch':      epoch,
            'model_state': model.state_dict(),
            'vocab':       VOCAB,
            'val_cer':    val_cer,
        }, SAVE_PATH)
        saved_str = ' ← best saved'
    else:
        saved_str = ''

    print(f'Epoch {epoch:3d}/{EPOCHS} | '
          f'train_loss={train_loss:.4f} | '
          f'val_loss={val_loss:.4f} | '
          f'CER={val_cer:.4f}{saved_str}')

    # Show sample predictions every 10 epochs
    if epoch % 10 == 0:
        print('  Sample predictions:')
        for p, t in zip(sample_preds, sample_tgts):
            print(f'    pred: "{p}"')
            print(f'    tgt:  "{t}"')
            print()

## 9. Evaluate & Plot

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('InkTransformer Training on MathWriting', fontsize=13)

epochs_x = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_x, history['train_loss'], label='Train Loss', color='#00d4aa')
ax1.plot(epochs_x, history['val_loss'],   label='Val Loss',   color='#ff6b35')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('CTC Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_x, history['val_cer'], label='Val CER', color='#a29bfe')
ax2.axhline(y=best_val_cer, linestyle='--', color='#ffd166',
            label=f'Best CER={best_val_cer:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('CER (lower = better)')
ax2.set_title('Character Error Rate'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best validation CER: {best_val_cer:.4f}')
print(f'Model saved to: {SAVE_PATH}')

## 10. Export to ONNX (for browser deployment)

In [ ]:
import json

# Load best checkpoint
checkpoint = torch.load(SAVE_PATH, map_location='cpu')
model.load_state_dict(checkpoint['model_state'])
model.eval()
model.cpu()

# ── Export to ONNX ──
ONNX_PATH = 'ink_model.onnx'

dummy_features    = torch.zeros(1, MAX_STROKES, MAX_PTS, 5)
dummy_stroke_mask = torch.ones(1, MAX_STROKES, MAX_PTS)
dummy_n_strokes   = torch.tensor([MAX_STROKES])

torch.onnx.export(
    model,
    (dummy_features, dummy_stroke_mask, dummy_n_strokes),
    ONNX_PATH,
    input_names  = ['features', 'stroke_mask', 'n_strokes'],
    output_names = ['log_probs'],
    dynamic_axes = {
        'features':    {0: 'batch', 1: 'n_strokes'},
        'stroke_mask': {0: 'batch', 1: 'n_strokes'},
        'log_probs':   {0: 'batch', 1: 'n_strokes'},
    },
    opset_version = 14,
    verbose       = False,
)

onnx_size_mb = os.path.getsize(ONNX_PATH) / 1e6
print(f'ONNX model saved: {ONNX_PATH}  ({onnx_size_mb:.1f} MB)')

# ── Save vocabulary ──
vocab_path = 'ink_vocab.json'
with open(vocab_path, 'w') as f:
    json.dump({'vocab': VOCAB, 'blank_idx': BLANK_IDX}, f)
print(f'Vocabulary saved: {vocab_path}')

print('\n✓ Ready for browser deployment with ONNX Runtime Web')
print('  Load ink_model.onnx + ink_vocab.json into ink-canvas-hierarchical.html')

## 11. Quick Inference Test
Test the model on a single sample before deploying.

In [ ]:
# Pick a random validation sample
sample_fp = random.choice(val_files)
s_strokes, s_label = parse_inkml(sample_fp)

# Preprocess
s_feats, s_mask = strokes_to_delta_features(s_strokes[:MAX_STROKES], MAX_PTS)
s_feats  = torch.FloatTensor(s_feats).unsqueeze(0)   # [1, S, P, 5]
s_mask   = torch.FloatTensor(s_mask).unsqueeze(0)    # [1, S, P]
s_ns     = torch.tensor([min(len(s_strokes), MAX_STROKES)])

# Run model
with torch.no_grad():
    log_probs = model(s_feats, s_mask, s_ns)
    prediction = ctc_greedy_decode(log_probs[0, :s_ns[0].item()], BLANK_IDX)

cer = compute_cer([prediction], [s_label])

print(f'Ground truth : "{s_label}"')
print(f'Prediction   : "{prediction}"')
print(f'CER          : {cer:.4f}  ({"PERFECT" if cer == 0 else "has errors"})')
print(f'Strokes used : {s_ns[0].item()}')